In [ ]:
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import gondola as gon


# =========================================================
# settings
# =========================================================
paddle_id_target = 1
Num = 200

use_flight = True
use_gaps = False

# =========================================================
# calibration + files
# =========================================================



if use_gaps:
    data_label_trigger = "Gaps"
else:
    data_label_trigger = "Track"





if use_flight:

    data_label = "Flight"

    calib = gon.calibration.load_rb_calibrations(
        Path(
            "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/calib/251222_010511UTC"
        )
    )
    
    files = [
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_10.251225_214949UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_100.251225_222655UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_102.251225_222745UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_110.251225_223103UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_111.251225_223129UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_112.251225_223154UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_114.251225_223243UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_115.251225_223308UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_117.251225_223357UTC.tof.gaps",
    ]
    
    
else:
    data_label = "Ground"

    calib = gon.calibration.load_rb_calibrations(
        Path(
            "/mnt/ucla-gaps-nas1/tof-data/antarctica/skua_hdd/data/calib/251123_215129UTC"
        )
    )

    files = [
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/skua_hdd/data/251/Run251_1345.251207_014209UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/skua_hdd/data/251/Run251_1848.251207_075922UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/skua_hdd/data/251/Run251_124.251206_101749UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/skua_hdd/data/251/Run251_1789.251207_071305UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/skua_hdd/data/251/Run251_860.251206_191532UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/skua_hdd/data/251/Run251_670.251206_165431UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/skua_hdd/data/251/Run251_86.251206_095058UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/skua_hdd/data/251/Run251_1323.251207_012610UTC.tof.gaps",
    ]


# =========================================================
# storage
# =========================================================

from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt

packcntEv = 0

# ---------------------------------------------------------
# storage: paddle_id -> list of times
# ---------------------------------------------------------
time_a_by_paddle = defaultdict(list)
time_b_by_paddle = defaultdict(list)

# =========================================================
# event loop
# =========================================================
for f in files:
    print(f"\nopening: {f}")
    
    reader = gon.io.TofPacketReader(
        str(f),
        filter=gon.packets.TofPacketType.TofEvent,
    )

    for pack in reader:
        ev = gon.events.TofEvent.from_bytestream(pack.payload, 0)
        packcntEv += 1
        
        if use_gaps:
            if str(ev.trigger_sources) == "[TriggerType.Track]":
                continue
        else:
            if str(ev.trigger_sources) == "[TriggerType.Gaps]":
                continue
        
        for rb in ev.rb_events:
            for hit in rb.hits:
                pid = int(hit.paddle_id)
                time_a_by_paddle[pid].append(hit.time_a)
                time_b_by_paddle[pid].append(hit.time_b)
                
# =========================================================
# clean arrays
# =========================================================
all_paddles = sorted(set(time_a_by_paddle.keys()) | set(time_b_by_paddle.keys()))

paddles = gon.db.TofPaddle.all()



for pid in all_paddles:
    time_a_by_paddle[pid] = np.asarray(time_a_by_paddle[pid], dtype=float)
    time_b_by_paddle[pid] = np.asarray(time_b_by_paddle[pid], dtype=float)

    time_a_by_paddle[pid] = time_a_by_paddle[pid][
        np.isfinite(time_a_by_paddle[pid])
    ]

    time_b_by_paddle[pid] = time_b_by_paddle[pid][
        np.isfinite(time_b_by_paddle[pid])
    ]


# =========================================================
# common bins
# =========================================================
all_times = []

for pid in all_paddles:
    all_times.append(time_a_by_paddle[pid])
    all_times.append(time_b_by_paddle[pid])

all_times = np.concatenate([x for x in all_times if len(x) > 0])

bins = np.linspace(
    np.nanmin(all_times),
    np.nanmax(all_times),
    100
)


# =========================================================
# plot 1: overlay every paddle, side A and side B
# =========================================================
fig, ax = plt.subplots(figsize=(12, 6))

for pid in all_paddles:
    ax.hist(
        time_a_by_paddle[pid],
        bins=bins,
        histtype="step",
        linewidth=1.2,
        alpha=0.65,
        label=f"Paddle {pid} A",
    )

    ax.hist(
        time_b_by_paddle[pid],
        bins=bins,
        histtype="step",
        linewidth=1.2,
        alpha=0.65,
        linestyle="--",
        label=f"Paddle {pid} B",
    )

ax.set_xlabel("time (ns)")
ax.set_ylabel("Counts")
ax.set_title(f"{data_label} {data_label_trigger}: Side A and B timing by paddle")
ax.set_yscale("log")
ax.grid(alpha=0.25)

# this can get crowded if there are many paddles
#ax.legend(loc="upper left", fontsize=8, ncol=2)

plt.tight_layout()
plt.show()


# =========================================================
# plot 2: total side A and total side B
# =========================================================

time_a_total = np.concatenate(
    [time_a_by_paddle[pid] for pid in all_paddles if len(time_a_by_paddle[pid])]
)

time_b_total = np.concatenate(
    [time_b_by_paddle[pid] for pid in all_paddles if len(time_a_by_paddle[pid]) > 0]
)

fig, ax = plt.subplots(figsize=(11, 5))

ax.hist(
    time_a_total,
    bins=bins,
    histtype="step",
    linewidth=1.8,
    label="Total side A",
)

ax.hist(
    time_b_total,
    bins=bins,
    histtype="step",
    linewidth=1.8,
    label="Total side B",
)

ax.set_xlabel("time (ns)")
ax.set_ylabel("Counts")
ax.set_title(f"{data_label} {data_label_trigger}: Total side A and side B timing")
ax.set_yscale("log")
ax.grid(alpha=0.25)
ax.legend(loc="upper left")

plt.tight_layout()
plt.show()

In [ ]:
panel_to_first_paddle = {}

for p in paddles:
    try:
        pid = int(p.paddle_id)
        panel = int(p.panel_id)
    except Exception:
        continue
        
    if pid not in all_paddles:
        continue

    # keep only first paddle for each panel
    if panel not in panel_to_first_paddle:
        panel_to_first_paddle[panel] = pid


selected_paddles = sorted(panel_to_first_paddle.values())

print("selected paddles:")
for panel, pid in sorted(panel_to_first_paddle.items()):
    print(f"panel {panel:2d} -> paddle {pid}")




time_a_total = np.concatenate([
    time_a_by_paddle[pid]
    for pid in selected_paddles
    if len(time_a_by_paddle[pid]) > 0
])

time_b_total = np.concatenate([
    time_b_by_paddle[pid]
    for pid in selected_paddles
    if len(time_b_by_paddle[pid]) > 0
])


# =========================================================
# highlighted paddle
# =========================================================

highlight_pid = 1

highlight_a = time_a_by_paddle[highlight_pid]
highlight_b = time_b_by_paddle[highlight_pid]


# =========================================================
# plot
# =========================================================

fig, ax = plt.subplots(figsize=(12, 6))


# ---------------------------------------------------------
# all representative paddles
# ---------------------------------------------------------
for pid in selected_paddles:

    ax.hist(
        time_a_by_paddle[pid],
        bins=bins,
        histtype="step",
        linewidth=1.0,
        alpha=0.45,
        color="orange",
    )

    ax.hist(
        time_b_by_paddle[pid],
        bins=bins,
        histtype="step",
        linewidth=1.0,
        alpha=0.45,
        linestyle="--",
        color="blue",
    )


# ---------------------------------------------------------
# total distributions
# ---------------------------------------------------------
ax.hist(
    time_a_total,
    bins=bins,
    histtype="step",
    linewidth=2.5,
    color="darkorange",
    label="Total side A",
)

ax.hist(
    time_b_total,
    bins=bins,
    histtype="step",
    linewidth=2.5,
    linestyle="--",
    color="darkblue",
    label="Total side B",
)


# ---------------------------------------------------------
# highlighted paddle 1
# ---------------------------------------------------------
ax.hist(
    highlight_a,
    bins=bins,
    histtype="step",
    linewidth=6,
    color="red",
    label=f"Paddle {highlight_pid} A",
)

ax.hist(
    highlight_b,
    bins=bins,
    histtype="step",
    linewidth=6,
    linestyle="--",
    color="cyan",
    label=f"Paddle {highlight_pid} B",
)


# =========================================================
# cosmetics
# =========================================================

ax.set_xlabel("time (ns)")
ax.set_ylabel("Counts")

ax.set_title(
    f"{data_label} {data_label_trigger}: Representative paddles by panel\n"
    f"(bold = paddle {highlight_pid})"
)

ax.set_yscale("log")


ax.legend(loc="upper left")

ax.set_xlim(0,350)
plt.tight_layout()
plt.show()

In [ ]:
''' for long boys... 
    files = [
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_10.251225_214949UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_100.251225_222655UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_102.251225_222745UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_110.251225_223103UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_111.251225_223129UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_112.251225_223154UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_114.251225_223243UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_115.251225_223308UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_117.251225_223357UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_118.251225_223423UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_119.251225_223447UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_120.251225_223513UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_122.251225_223602UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_123.251225_223627UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_125.251225_223717UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_127.251225_223807UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_129.251225_223856UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_13.251225_215103UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_130.251225_223921UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_131.251225_223945UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_136.251225_224152UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_137.251225_224217UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_138.251225_224242UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_140.251225_224332UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_141.251225_224357UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_142.251225_224422UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_143.251225_224448UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_15.251225_215153UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_16.251225_215217UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_18.251225_215307UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_19.251225_215331UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_21.251225_215420UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_22.251225_215445UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_24.251225_215533UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_27.251225_215647UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_30.251225_215800UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_33.251225_215914UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_36.251225_220028UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_37.251225_220053UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_38.251225_220118UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_41.251225_220232UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_44.251225_220346UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_45.251225_220411UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_52.251225_220704UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_46.251225_220436UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_5.251225_214745UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_50.251225_220614UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_51.251225_220639UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_57.251225_220908UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_54.251225_220754UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_55.251225_220819UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_56.251225_220844UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_6.251225_214810UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_61.251225_221046UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_62.251225_221111UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_66.251225_221251UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_63.251225_221136UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_67.251225_221316UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_70.251225_221431UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_7.251225_214835UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_76.251225_221702UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_78.251225_221751UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_77.251225_221727UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_79.251225_221816UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_82.251225_221931UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_8.251225_214900UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_83.251225_221956UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_99.251225_222631UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_84.251225_222020UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_88.251225_222159UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_89.251225_222224UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_91.251225_222314UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_95.251225_222452UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_96.251225_222517UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_128.251225_223831UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_28.251225_215711UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_103.251225_222810UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_26.251225_215623UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_12.251225_215039UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_106.251225_222925UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_101.251225_222720UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_2.251225_214632UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_20.251225_215356UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_32.251225_215849UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_3.251225_214656UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_126.251225_223742UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_121.251225_223537UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_0.251225_214542UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_109.251225_223038UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_40.251225_220208UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_108.251225_223013UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_1.251225_214607UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_139.251225_224307UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_104.251225_222835UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_25.251225_215558UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_4.251225_214721UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_135.251225_224126UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_132.251225_224010UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_23.251225_215509UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_43.251225_220321UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_116.251225_223332UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_31.251225_215825UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_29.251225_215736UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_107.251225_222949UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_14.251225_215128UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_39.251225_220143UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_34.251225_215939UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_113.251225_223218UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_11.251225_215014UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_42.251225_220257UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_47.251225_220500UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_134.251225_224101UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_105.251225_222900UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_133.251225_224036UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_124.251225_223652UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_35.251225_220004UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_17.251225_215242UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_48.251225_220525UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_49.251225_220549UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_58.251225_220932UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_53.251225_220729UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_60.251225_221022UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_59.251225_220957UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_64.251225_221201UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_71.251225_221456UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_65.251225_221226UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_68.251225_221341UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_69.251225_221406UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_81.251225_221906UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_72.251225_221521UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_73.251225_221546UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_74.251225_221611UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_75.251225_221636UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_80.251225_221841UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_94.251225_222427UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_85.251225_222045UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_86.251225_222110UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_87.251225_222134UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_9.251225_214925UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_90.251225_222249UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_92.251225_222338UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_93.251225_222403UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_97.251225_222542UTC.tof.gaps",
        "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_98.251225_222606UTC.tof.gaps",
    ]'''